# Space-map — Xenium with gene expression (multi-channel)

One-click **multi-channel** registration driven by density, with **gene
expression** channels (and optionally cell types) weighting the affine
selection. This notebook shows the input helpers in `space_map.io` that turn
a 10x / Xenium sparse expression matrix into the wide table `init_df`
consumes — the initialization path itself is unchanged.

**Data:**
- a coordinate CSV with `x`, `y`, a cell-id column, `layer`, and (optional)
  `cell_type` — e.g. `xenium_polyp.csv.gz`
- one 10x matrix directory **per layer** (`matrix.mtx.gz`, `features.tsv.gz`,
  `barcodes.tsv.gz`)

Set the paths below and run all cells.

In [ ]:
import os, sys
# Ensure this repo's space_map wins over any editable/site-packages install.
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import space_map
from space_map import Slice
from space_map.flow import FlowImport, FlowExport, AutoFlowMultiCenter4

# --- edit these to your local files ---
COORDS = 'data/xenium_polyp.csv.gz'
# matrix directory for each layer id, in slice order:
MATRIX_DIRS = {
    1: 'data/matrix/1',
    # 2: 'data/matrix/2', 3: 'data/matrix/3', ...
}
CELL_ID_COL, X_COL, Y_COL = 'cell_id', 'x', 'y'
CELLTYPE_COL = 'cell_type'
GENES = ['EPCAM', 'VIM', 'ACTA2']   # or set GENES=None and TOP_N=<int>
TOP_N_GENES = None
WORKDIR = 'out/xenium_expression/_work'

## Build a wide table per layer from the expression matrix

`space_map.io.load_10x_matrix` reads a matrix directory into a
cells×genes table; `build_wide_df` joins it onto the layer's coordinates by
cell id and keeps the chosen genes. It renames the cell-type column to
`celltype` so `init_df` auto-expands it to one-hot channels.

In [ ]:
coords = pd.read_csv(COORDS, usecols=[X_COL, Y_COL, CELL_ID_COL, 'layer', CELLTYPE_COL])

layers = []
for layer_id in sorted(MATRIX_DIRS):
    expr = space_map.io.load_10x_matrix(MATRIX_DIRS[layer_id])
    c = coords[coords['layer'] == layer_id]
    wide = space_map.io.build_wide_df(
        c, expr, cell_id_col=CELL_ID_COL, x_col=X_COL, y_col=Y_COL,
        celltype_col=CELLTYPE_COL, genes=GENES, top_n=TOP_N_GENES)
    layers.append(wide)
    print(f'layer {layer_id}: {wide.shape[0]} cells, columns = {list(wide.columns)}')

## Import (multi-channel) — density + cell types + gene channels

`init_multi` normalizes coordinates from density and applies the same shift
to every channel. Each gene becomes a `gene::<name>` channel; each kept cell
type becomes a `ct::<type>` channel.

In [ ]:
fi = FlowImport(WORKDIR)
ids = [str(k) for k in sorted(MATRIX_DIRS)]
slices = fi.init_multi(layers, ids=ids, celltype_col='celltype', top_n=10)
s0 = slices[0]
print(f'{len(slices)} slices; channels on slice0:')
print('  density :', 'DF' in s0.imgs)
print('  genes   :', sorted(k for k in s0.imgs if k.startswith('gene::')))
print('  types   :', sorted(k for k in s0.imgs if k.startswith('ct::')))

## Affine (density features, expression/cell-type-weighted) + non-rigid

With a single layer there is no pair to register; add more entries to
`MATRIX_DIRS` (and their coordinates) to align a real stack.

In [ ]:
KEY = Slice.align1Key
if len(slices) > 1:
    af = AutoFlowMultiCenter4(slices, initJKey=Slice.rawKey, alignMethod='auto')
    af.affine(useKey='DF', show=False)
    af.ldm_pair(Slice.align1Key, Slice.align2Key, show=False)
    KEY = Slice.align2Key
    print('registration done')
else:
    for s in slices:
        s.applyH(Slice.rawKey, None, KEY)  # identity: single layer
    print('single layer: nothing to align (channels still exported below)')

## Export all channels, per-channel 3D-volume TIFFs, and a 3D plot

In [ ]:
fe = FlowExport(slices)
table = fe.export_all_channels(KEY, path='out/xenium_expression/all_channels.csv')
print('exported', table.shape, 'rows'); print(table.groupby('channel').size())

# Imaris: per-layer, per-channel single-channel TIFFs (1000x1000)
#   -> out/xenium_expression/imaris/  named c_<layer>_<channel>.tiff
fe.export_imaris_tiff('out/xenium_expression/imaris', key=KEY, size=1000)

# 3D reconstruction colored by cell type (~45 deg)
fe.plot_3d(key=KEY, save='out/xenium_expression/reconstruction_3d.png',
           title='Xenium - 3D reconstruction')
print('wrote imaris/ and reconstruction_3d.png')